# Clip Farm — Google Colab

Run the Clip Farm VOD workflow in Colab: set up the tools, scan a stream VOD, let an OpenRouter free model find moments, review the shortlist, render vertical clips, and download the results.

**Before you start:** use a Colab GPU runtime if available (Runtime → Change runtime type). You need an OpenRouter API key; the notebook defaults to the `openrouter/free` model router. Add `OPENROUTER_API_KEY` under Colab Secrets (key icon) or enter it privately when prompted. A free model is not guaranteed to be available at all times and may be slower or less consistent than a paid model.

The notebook uses temporary Colab storage. Download your results before the runtime disconnects. Use VODs you have permission to clip, and watch every clip before posting. Live-follow mode is omitted because Colab runtimes can disconnect and do not reliably keep background recorders running.

## 1. Install Colab dependencies

This replaces the Mac/Homebrew setup. It installs FFmpeg, Python libraries, Deno (for supported yt-dlp extraction), and builds Whisper.cpp.

In [ ]:
import os, pathlib, subprocess, sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "cmake", "build-essential", "git", "pkg-config"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "opencv-python-headless", "pillow", "yt-dlp[default]"], check=True)

# Install Deno using its official installer.
subprocess.run(["bash", "-lc", "curl -fsSL https://deno.land/install.sh | sh"], check=True)
os.environ["PATH"] = f"{pathlib.Path.home() / '.deno' / 'bin'}:{os.environ['PATH']}"

whisper_src = pathlib.Path("/content/whisper.cpp")
if not whisper_src.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/whisper.cpp.git", str(whisper_src)], check=True)
subprocess.run(["cmake", "-S", str(whisper_src), "-B", str(whisper_src / "build"), "-DCMAKE_BUILD_TYPE=Release", "-DWHISPER_BUILD_TESTS=OFF", "-DWHISPER_BUILD_EXAMPLES=ON"], check=True)
subprocess.run(["cmake", "--build", str(whisper_src / "build"), "--config", "Release", "-j", "2"], check=True)
whisper_bins = list((whisper_src / "build").rglob("whisper-cli"))
if not whisper_bins:
    raise FileNotFoundError("Whisper.cpp built without a whisper-cli executable")
subprocess.run(["ln", "-sf", str(whisper_bins[0]), "/usr/local/bin/whisper-cli"], check=True)
print("Dependencies installed. FFmpeg:", subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])

## 2. Upload the Clip Farm project ZIP

Upload the project ZIP you downloaded. The notebook locates `scripts/cf.py` and the workflow references inside it.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Please upload the Clip Farm project .zip file")
source_root = pathlib.Path("/content/clipfarm-source")
source_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(source_root)
cf_candidates = list(source_root.rglob("scripts/cf.py"))
skill_candidates = list(source_root.rglob("reference/picking.md"))
if not cf_candidates or not skill_candidates:
    raise FileNotFoundError("The uploaded ZIP does not contain the Clip Farm scripts and references")
CF_SCRIPT = cf_candidates[0]
SKILL_DIR = skill_candidates[0].parent.parent
print("Project:", SKILL_DIR)

## 3. Download the Whisper model and initialize Clip Farm

The `small.en` model is about 480 MB. Chunk length is set to six minutes so transcript-picking requests fit more comfortably within free-model limits. For a very long VOD, the scan and transcription may take a while.

In [ ]:
import sys, urllib.request

os.environ["CLIPFARM_HOME"] = "/content/.clip-farm"
os.environ["CLIPFARM_WORK"] = "/content/ClipFarm"
sys.path.insert(0, str(CF_SCRIPT.parent))
from clipfarm import util

util.MODELS.mkdir(parents=True, exist_ok=True)
whisper_model = util.MODELS / "ggml-small.en.bin"
if not whisper_model.exists() or whisper_model.stat().st_size < 10_000_000:
    urllib.request.urlretrieve("https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-small.en.bin", whisper_model)
    if whisper_model.stat().st_size < 10_000_000:
        raise RuntimeError("Whisper model download was incomplete")

cfg = util.settings()
cfg["chunk_minutes"] = 6
util.save_settings(cfg)

# Optional: download the face detector model; rendering falls back if unavailable.
yunet = util.MODELS / "face_detection_yunet_2023mar.onnx"
if not yunet.exists():
    try:
        urllib.request.urlretrieve("https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx", yunet)
    except Exception as exc:
        print("Face detector download skipped; renderer will use its fallback:", exc)
print("Ready. Work and output folder:", util.WORK)

## 4. Set the OpenRouter key and model

Create a key at [OpenRouter](https://openrouter.ai/keys). Colab Secrets is preferred; otherwise the key is read with a hidden prompt. It is not written into the notebook or Clip Farm files. `openrouter/free` routes to an available free model; you can change `OPENROUTER_MODEL` to another free model ID listed by OpenRouter.

In [ ]:
import getpass, requests

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = None
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("OpenRouter API key: ")
if not OPENROUTER_API_KEY.strip():
    raise ValueError("An OpenRouter API key is required")
OPENROUTER_MODEL = "openrouter/free"
print("Configured model:", OPENROUTER_MODEL)

## 5. (Optional) Find streams, then scan a VOD

Run the optional cell to list streams, or skip it and provide a VOD URL below. Age-restricted VODs may require cookies; browser-cookie extraction is generally not available from Colab, so use a VOD accessible without login.

In [ ]:
def cf(*args):
    cmd = [sys.executable, str(CF_SCRIPT), *map(str, args)]
    return subprocess.run(cmd, check=True, text=True, env=os.environ.copy())

# Optional stream discovery. Uncomment and run to list current results.
# cf("find", "--game", "Grand Theft Auto VI", "--limit", "10")

In [ ]:
from clipfarm import stream
from clipfarm.util import stream_dir

STREAM_URL = input("Paste a YouTube, Twitch, or Kick VOD URL: ").strip()
if not STREAM_URL:
    raise ValueError("A VOD URL is required")
STREAM_KEY = stream.scan(STREAM_URL)
STREAM_DIR = stream_dir(STREAM_KEY)
print("Scanned stream:", STREAM_KEY, "\nStream files:", STREAM_DIR)

## 6. AI-pick moments in every transcript chunk

This follows `reference/picking.md`: favor self-contained moments with a payoff, use the signal peaks as clues, and leave weak chunks empty. The notebook saves each response in the expected `picks/chunk_NN.json` format. Existing pick files are left untouched so you can resume or edit them; set `FORCE_REPICK = True` to replace them.

In [ ]:
import json, re

FORCE_REPICK = False
PICKING_RULES = (SKILL_DIR / "reference" / "picking.md").read_text()
chunk_files = sorted((STREAM_DIR / "chunks").glob("chunk_*.md"))
if not chunk_files:
    raise FileNotFoundError("No transcript chunks were produced by the scan")

def openrouter_json(system_text, user_text, max_tokens=3000):
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json",
                 "HTTP-Referer": "https://colab.research.google.com", "X-Title": "Clip Farm Colab"},
        json={"model": OPENROUTER_MODEL, "temperature": 0.2, "max_tokens": max_tokens,
              "messages": [{"role": "system", "content": system_text}, {"role": "user", "content": user_text}]},
        timeout=240,
    )
    if not response.ok:
        raise RuntimeError(f"OpenRouter request failed ({response.status_code}): {response.text[:1200]}")
    payload = response.json()
    try:
        text = payload["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise RuntimeError(f"OpenRouter returned no assistant text: {payload}") from exc
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError("OpenRouter returned an empty response")
    text = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text.strip(), flags=re.IGNORECASE)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        if start < 0:
            raise RuntimeError(f"Model did not return a JSON object: {text[:1000]}")
        try:
            return json.JSONDecoder().raw_decode(text[start:])[0]
        except json.JSONDecodeError as exc:
            raise RuntimeError(f"Model returned invalid JSON: {text[:1000]}") from exc

PICK_SYSTEM = "You select strong, standalone short-form video moments from stream transcripts. Follow the supplied picking rules exactly. Return only a JSON object with a moments array. Do not invent events, timestamps, or signals."
picks_dir = STREAM_DIR / "picks"
picks_dir.mkdir(exist_ok=True)
for chunk_path in chunk_files:
    out_path = picks_dir / f"{chunk_path.stem}.json"
    if out_path.exists() and not FORCE_REPICK:
        print("Keeping existing:", out_path.name)
        continue
    chunk_no = int(chunk_path.stem.split("_")[-1])
    prompt = (f"PICKING RULES:\n{PICKING_RULES}\n\nTRANSCRIPT CHUNK:\n{chunk_path.read_text()}\n\n"
              f"Return exactly {{\"chunk\": {chunk_no}, \"moments\": [ ... ]}}. "
              "Each moment needs start, end, peak, type, integer score (0-100), standalone, title, and why. "
              "Use stream times shown in the transcript. Return an empty moments list if nothing is strong enough.")
    result = openrouter_json(PICK_SYSTEM, prompt)
    moments = result.get("moments") if isinstance(result, dict) else None
    if not isinstance(moments, list):
        raise ValueError(f"{chunk_path.name}: model response has no moments list: {result}")
    result["chunk"] = chunk_no
    out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\n")
    print(f"{chunk_path.name}: saved {len(moments)} moment(s)")
    
print("All chunk picks are present. You can inspect/edit JSON files in:", picks_dir)

## 7. Validate picks and build the shortlist

Run the project's own checks and scoring, which combine AI scores with the signal data and remove duplicate or previously clipped moments.

In [ ]:
cf("check", STREAM_KEY)
cf("shortlist", STREAM_KEY, "--top", "12")
shortlist_path = STREAM_DIR / "shortlist.md"
print(shortlist_path.read_text())

## 8. Choose final clips with the free model

The model applies `reference/final-round.md`, keeps no more than the configured maximum, and writes the expected `final.json`. Review that file before rendering if you want to change the selection. Clips with music can be claimed or muted; check the warnings in the shortlist.

In [ ]:
FINAL_RULES = (SKILL_DIR / "reference" / "final-round.md").read_text()
shortlist_text = shortlist_path.read_text()
FINAL_SYSTEM = "You are a strict short-form video editor. Pick only moments that pass the supplied final-round rules. Use candidate IDs and timestamps exactly as provided. Return only a JSON object with a clips array."
final_result = openrouter_json(
    FINAL_SYSTEM,
    f"FINAL-ROUND RULES:\n{FINAL_RULES}\n\nSHORTLIST:\n{shortlist_text}\n\n"
    "Return exactly {\"clips\": [...]} with each chosen candidate's id, start, end, peak, type, score, hook, caption, hashtags, and why. "
    "Use at most the configured default number of clips (3), never more than 5, and return fewer if quality is weak. "
    "Use only IDs/times from the shortlist. Output an empty clips list if none qualify.",
    max_tokens=4000,
)
final_clips = final_result.get("clips") if isinstance(final_result, dict) else None
if not isinstance(final_clips, list):
    raise ValueError(f"Final response has no clips list: {final_result}")
candidates = {item["id"]: item for item in json.loads((STREAM_DIR / "shortlist.json").read_text())}
max_final = min(util.settings()["clips_per_stream"], util.settings()["max_clips_per_stream"], 5)
if len(final_clips) > max_final:
    raise ValueError(f"Model chose {len(final_clips)} clips; the configured limit is {max_final}. Edit final.json and retry")
seen_ids = set()
for item in final_clips:
    if not isinstance(item, dict) or item.get("id") not in candidates:
        raise ValueError(f"Final response contains an unknown candidate: {item}")
    if item["id"] in seen_ids:
        raise ValueError(f"Final response repeats candidate {item['id']}")
    seen_ids.add(item["id"])
    candidate = candidates[item["id"]]
    start_s, end_s = util.parse_t(item["start"]), util.parse_t(item["end"])
    peak_s = util.parse_t(item["peak"])
    if start_s < candidate["start"] - 1 or end_s > candidate["end"] + 1 or end_s <= start_s:
        raise ValueError(f"{item['id']} times fall outside the shortlisted candidate; edit the response and retry")
    if not start_s <= peak_s <= end_s:
        raise ValueError(f"{item['id']} peak must be inside the clip")
    if not util.settings()["min_clip_s"] <= end_s - start_s <= util.settings()["max_clip_s"]:
        raise ValueError(f"{item['id']} duration is outside the configured clip limits")
    item["start"], item["end"], item["peak"] = util.hms(start_s), util.hms(end_s), util.hms(peak_s)
    item["type"], item["score"] = candidate["type"], candidate["score"]
    if not item.get("hook") or not item.get("caption") or not isinstance(item.get("hashtags"), list) or not item["hashtags"]:
        raise ValueError(f"Final response is missing hook, caption, or hashtags: {item}")
final_path = STREAM_DIR / "final.json"
final_path.write_text(json.dumps({"clips": final_clips}, indent=2, ensure_ascii=False) + "\n")
print(final_path.read_text())
print("Review/edit this file before rendering:", final_path)

## 9. Render, review, and download

Rendering downloads only the selected sections, creates vertical clips, and builds the review page. This can take several minutes per clip.

In [ ]:
cf("render", STREAM_KEY)
cf("review", "--no-open")
clip_dir = util.WORK / "clips" / util.today()
print("Review page and clips:", clip_dir)
print("Files:", *[p.name for p in sorted(clip_dir.iterdir())], sep="\n - ")

In [ ]:
import shutil

download_zip = pathlib.Path("/content/ClipFarm_results.zip")
shutil.make_archive(str(download_zip.with_suffix("")), "zip", root_dir=clip_dir)
files.download(str(download_zip))

## Optional: learn from your posted Shorts

After posting and collecting results, set your channel URL below to compare performance against your clips. This needs the channel's Shorts to be publicly readable.

In [ ]:
# CHANNEL_URL = "https://www.youtube.com/@yourchannel"
# if CHANNEL_URL:
#     cf("learn", CHANNEL_URL)